Создание файла для получения информации о фильмах.

In [ ]:
import requests
import json
import Settings

url = "https://api.kinopoisk.dev/v1.4/movie?page=1&limit=250&selectFields=id&selectFields=externalId&selectFields=name&selectFields=enName&selectFields=alternativeName&selectFields=names&selectFields=description&selectFields=shortDescription&selectFields=slogan&selectFields=type&selectFields=typeNumber&selectFields=isSeries&selectFields=status&selectFields=year&selectFields=releaseYears&selectFields=rating&selectFields=ratingMpaa&selectFields=ageRating&selectFields=votes&selectFields=seasonsInfo&selectFields=budget&selectFields=audience&selectFields=movieLength&selectFields=seriesLength&selectFields=totalSeriesLength&selectFields=genres&selectFields=countries&selectFields=poster&selectFields=backdrop&selectFields=logo&selectFields=ticketsOnSale&selectFields=videos&selectFields=networks&selectFields=persons&selectFields=facts&selectFields=fees&selectFields=premiere&selectFields=similarMovies&selectFields=sequelsAndPrequels&selectFields=watchability&selectFields=lists&selectFields=top10&selectFields=top250&selectFields=updatedAt&sortField=&sortType=1&lists=top250"

headers = {
    "accept": "application/json",
    "X-API-KEY": Settings.XAPIKEY
}

response = requests.get(url, headers=headers)

print(response.text)

response_json = json.dumps(response.text, indent=4)
print(response_json)
with open('all_film_list.json', 'w') as file:
    # Записываем JSON-строку в файл
    file.write(response_json)


Создание БД и загрузка информации и 1234.json

In [14]:
import sqlite3
import json

# Подключение к базе данных SQLite
conn = sqlite3.connect('movies.db')
cursor = conn.cursor()

# Создание таблицы для фильмов
cursor.execute('''
CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY,
    name TEXT,
    alternative_name TEXT,
    en_name TEXT,
    type TEXT,
    type_number INTEGER,
    year INTEGER,
    description TEXT,
    short_description TEXT,
    slogan TEXT,
    status TEXT,
    rating_kp REAL,
    rating_imdb REAL,
    rating_film_critics REAL,
    rating_russian_film_critics REAL,
    rating_await REAL,
    votes_kp INTEGER,
    votes_imdb INTEGER,
    votes_film_critics INTEGER,
    votes_russian_film_critics INTEGER,
    votes_await INTEGER,
    movie_length INTEGER,
    age_rating INTEGER,
    rating_mpaa TEXT
)
''')

# Создание таблицы для жанров
cursor.execute('''
CREATE TABLE IF NOT EXISTS genres (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    movie_id INTEGER,
    genre_name TEXT,
    FOREIGN KEY (movie_id) REFERENCES movies(id)
)
''')

# Создание таблицы для стран
cursor.execute('''
CREATE TABLE IF NOT EXISTS countries (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    movie_id INTEGER,
    country_name TEXT,
    FOREIGN KEY (movie_id) REFERENCES movies(id)
)
''')

# Создание таблицы для актеров
cursor.execute('''
CREATE TABLE IF NOT EXISTS persons (
    id INTEGER PRIMARY KEY,
    movie_id INTEGER,
    name TEXT,
    en_name TEXT,
    description TEXT,
    profession TEXT,
    en_profession TEXT,
    photo TEXT,
    FOREIGN KEY (movie_id) REFERENCES movies(id)
)
''')

# Создание таблицы для информации о доступности просмотра
cursor.execute('''
CREATE TABLE IF NOT EXISTS watchability (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    movie_id INTEGER,
    service_name TEXT,
    link TEXT,
    FOREIGN KEY (movie_id) REFERENCES movies(id)
)
''')

# Создание таблицы для информации о постерах
cursor.execute('''
CREATE TABLE IF NOT EXISTS posters (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    movie_id INTEGER,
    url TEXT,
    preview_url TEXT,
    FOREIGN KEY (movie_id) REFERENCES movies(id)
)
''')

# Создание таблицы для пользователей
cursor.execute('''
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    username TEXT,
    language_code TEXT,
    is_bot INTEGER,
    registration_date TEXT,
    last_activity_date TEXT,
    birth_date TEXT
)
''')

# Создание таблицы для действий пользователей
cursor.execute('''
CREATE TABLE IF NOT EXISTS actions (
    action_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    movie_id INTEGER,
    want_to_watch INTEGER,
    rating INTEGER,
    FOREIGN KEY (user_id) REFERENCES users(user_id),
    FOREIGN KEY (movie_id) REFERENCES movies(id)
)
''')
# Загрузка данных из JSON
with open('film_list.json', 'r', encoding='utf-8') as file:
    data = json.load(file)
    movies = data['docs']
    
    for movie in movies:
        # Вставка данных в таблицу movies
        cursor.execute('''
        INSERT OR IGNORE INTO movies (
            id, name, alternative_name, en_name, type, type_number, year,
            description, short_description, slogan, status, rating_kp,
            rating_imdb, rating_film_critics, rating_russian_film_critics,
            rating_await, votes_kp, votes_imdb, votes_film_critics,
            votes_russian_film_critics, votes_await, movie_length,
            age_rating, rating_mpaa
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            movie['id'], movie['name'], movie.get('alternativeName'), movie.get('enName'),
            movie['type'], movie['typeNumber'], movie['year'],
            movie.get('description'), movie.get('shortDescription'), movie.get('slogan'),
            movie.get('status'), movie.get('rating', {}).get('kp'),
            movie.get('rating', {}).get('imdb'), movie.get('rating', {}).get('filmCritics'),
            movie.get('rating', {}).get('russianFilmCritics'), movie.get('rating', {}).get('await'),
            movie.get('votes', {}).get('kp'), movie.get('votes', {}).get('imdb'),
            movie.get('votes', {}).get('filmCritics'), movie.get('votes', {}).get('russianFilmCritics'),
            movie.get('votes', {}).get('await'), movie.get('movieLength'),
            movie.get('ageRating'), movie.get('ratingMpaa')
        ))
        
        # Вставка данных в таблицу genres
        for genre in movie.get('genres', []):
            cursor.execute('''
            INSERT OR IGNORE INTO genres (movie_id, genre_name) VALUES (?, ?)
            ''', (movie['id'], genre['name']))
        
        # Вставка данных в таблицу countries
        for country in movie.get('countries', []):
            cursor.execute('''
            INSERT OR IGNORE INTO countries (movie_id, country_name) VALUES (?, ?)
            ''', (movie['id'], country['name']))
        
        # Вставка данных в таблицу persons
        for person in movie.get('persons', []):
            cursor.execute('''
            INSERT OR IGNORE INTO persons (id, movie_id, name, en_name, description, profession, en_profession, photo)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                person['id'], movie['id'], person.get('name'), person.get('enName'),
                person.get('description'), person.get('profession'), person.get('enProfession'),
                person.get('photo')
            ))

        # Вставка данных в таблицу watchability
        for item in movie.get('watchability', {}).get('items', []):
            cursor.execute('''
            INSERT OR IGNORE INTO watchability (movie_id, service_name, link)
            VALUES (?, ?, ?)
            ''', (movie['id'], item.get('name'), item.get('url')))
        
        # Вставка данных в таблицу posters
        poster_data = movie.get('poster', {})
        cursor.execute('''
        INSERT INTO posters (movie_id, url, preview_url)
        VALUES (?, ?, ?)
        ''', (
            movie['id'],
            poster_data.get('url', ''),
            poster_data.get('previewUrl', '')
        ))

# Сохранение изменений и закрытие соединения
conn.commit()
conn.close()


In [2]:
import sqlite3
def save_user_info(user_info):
    conn = sqlite3.connect('movies.db')  # Подключение к базе данных 'movies.db'
    cursor = conn.cursor()
    cursor.execute('''INSERT OR REPLACE INTO users 
                      (user_id, first_name, last_name, username, language_code, is_bot, birth_date, registration_date, last_activity_date) 
                      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                   (user_info['user_id'], user_info['first_name'], user_info.get('last_name'), user_info.get('username'), 
                    user_info.get('language_code'), user_info['is_bot'], user_info.get('birth_date'), 
                    user_info['registration_date'], user_info['last_activity_date']))
    conn.commit()
    conn.close()
    
user_info = {'user_id': 280245855, 'first_name': 'Павел', 'last_name': None, 'username': 'itsmyacc', 'language_code': 'ru', 'is_bot': False, 'birth_date': None, 'registration_date': '2024-08-08 12:12:59', 'last_activity_date': '2024-08-08 12:12:59'}
save_user_info(user_info)